# 5章 構造化データの出力 (1)

## 事前準備

当サンプルコードを実行するには、OpenAIのキーが必要です。

### OpenAI
- OpenAIのプラットフォーム（ https://platform.openai.com/ ）にアクセスし、ご自身のアカウントでログインして、APIキーを生成してください。
- 生成したAPIキーを定数 `OPENAI_API_KEY` に設定します。

```
OPENAI_API_KEY=<作成したAPIキー>
```



In [ ]:
OPENAI_API_KEY= "<作成したAPIキー>"

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
!wget "https://drive.google.com/uc?export=download&id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_" -O apple.txt

--2026-10-22 06:16:35--  https://drive.google.com/uc?export=download&id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_
Resolving drive.google.com (drive.google.com)... 172.253.153.139, 172.253.153.102, 172.253.153.138, ...
Connecting to drive.google.com (drive.google.com)|172.253.153.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_&export=download [following]
--2026-10-22 06:16:35--  https://drive.usercontent.google.com/download?id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 172.253.154.132, 2a00:1450:4013:c24::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|172.253.154.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 410321 (401K) [application/octet-stream]
Saving to: ‘apple.txt’

apple.txt           100%[===================>] 400.70K  --.-KB/s

In [ ]:
MAX_LENGTH = 100_000
# サンプルの10-K SEC提出書類からテキストを読み込む
with open('apple.txt', 'r') as file:
    sec_filing = file.read()
sec_filing = sec_filing[:MAX_LENGTH]

## 推論時の制約技術

### プロンプトエンジニアリング

In [ ]:
from openai import OpenAI
client = OpenAI()

In [ ]:
prompt = """
以下のテキストに含まれる主要な財務データについて、2人の人物による会話をJSON形式で生成してください。

<JSON_FORMAT>
{
   "Person1": {
     "name": "Alice",
     "statement": "第1四半期の売上高は、前年同期比で20%増加しました。"
   },
   "Person2": {
     "name": "Bob",
     "statement": "それは素晴らしいニュースですね、純利益率についてはどうですか。"
   }
}
</JSON_FORMAT>
"""

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": sec_filing}
    ]
)

In [ ]:
response_content = response.choices[0].message.content
print(response_content)


{
  "Person1": {
    "name": "Alice",
    "statement": "Appleの2024年度の通期売上高は3,910億ドルでした。第4四半期の売上高は949億ドルで、前年同期比6%増です。"
  },
  "Person2": {
    "name": "Bob",
    "statement": "すごいですね。では、2024年度の純利益や自己株式取得の状況はどうなっていますか。"
  }
}


In [ ]:
import json

def is_json(myjson):
  try:
    json.loads(myjson)
    return True
  except ValueError as e:
    return False

is_json(response_content)

True

In [ ]:
import json
content = response.choices[0].message.content
data = json.loads(content)

In [ ]:
from IPython.display import JSON
JSON(data)

<IPython.core.display.JSON object>

### JSONモード

In [ ]:
prompt = f"""
以下のテキストに含まれる主要な財務データについて、2人の人物による会話をJSON形式で生成してください。
TEXT: {sec_filing}
"""
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt}],
    response_format = { "type": "json_object" }
)

In [ ]:
response_content = response.choices[0].message.content
print(response_content)

{
  "dialogue": [
    {
      "speaker": "A",
      "text": "Appleの2024年度10-Kから主要な財務・株主関連データを整理すると、まず自社株買いが目立ちます。2024年6月30日〜9月28日の3か月で、合計112,260千株を平均$223.34前後で買い戻しています。"
    },
    {
      "speaker": "B",
      "text": "その期間の内訳は、6/30〜8/3が35,697千株で平均$224.11、8/4〜8/31が42,910千株で平均$221.39、9/1〜9/28が33,653千株で平均$222.86ですね。かなり積極的な資本還元です。"
    },
    {
      "speaker": "A",
      "text": "加えて、2024年9月28日時点で、2024年5月2日に承認された自社株買いプログラムの上限は$1100億でしたが、そのうち$209億が消化済みと開示されています。"
    },
    {
      "speaker": "B",
      "text": "つまり、まだ大きな残枠がある一方、同時に前回の2023年5月のプログラムも2024年4Qで最後の$41億を使い切ったということですね。Appleは配当と自社株買いの両方で株主還元を継続している印象です。"
    },
    {
      "speaker": "A",
      "text": "株主数も開示されていて、2024年10月18日時点での登録株主数は23,301人でした。市場での流動性が非常に高い銘柄なので、これは登録株主ベースとしては納得感があります。"
    },
    {
      "speaker": "B",
      "text": "株価パフォーマンスも強いです。5年間の累積トータルリターンは、Appleが$100投資で$430まで成長しています。比較すると、S&P 500は$210、Dow Jones U.S. Technology Supersector Indexは$322です。"
    },
    {
      "speaker": "A",
      "text": "つまり、Appleは指数を

### 構造化出力とPydantic

In [ ]:
from pydantic import BaseModel

class SECExtraction(BaseModel):
    mentioned_entities: list[str]
    mentioned_places: list[str]

In [ ]:
from openai import OpenAI

def extract_from_sec_filing(sec_filing_text: str, prompt: str) -> SECExtraction:
  """
  入力されたSEC提出書類のテキストから構造化データを抽出する。
  """
  client = OpenAI()
  completion = client.beta.chat.completions.parse(
      model="gpt-5.4-mini",
      messages=[
          {
              "role": "system",
              "content": prompt
          },
          {"role": "user", "content": sec_filing_text}
      ],
      response_format=SECExtraction
  )
  return completion.choices[0].message.parsed

prompt_extraction = '''あなたは構造化データ抽出の専門家です。
SEC提出書類から得た非構造化テキストが与えられるので、
そこに記載されている企業・組織などのエンティティ名と場所を、
指定された構造に抽出してください。'''

sec_extraction = extract_from_sec_filing(sec_filing, prompt_extraction)

In [ ]:
print("Extracted entities:", sec_extraction.mentioned_entities)
print("Extracted places:", sec_extraction.mentioned_places)

Extracted entities: ['Apple Inc.', 'The Company', 'Apple', 'Apple Watch Ultra 2', 'Apple Watch Series 10', 'Apple Watch SE', 'Beats', 'Apple Vision Pro', 'Apple TV', 'HomePod', 'HomePod mini', 'AppleCare', 'Apple Arcade', 'Apple Fitness+', 'Apple Music', 'Apple News+', 'Apple TV+', 'Apple Card', 'Apple Pay', 'iPhone', 'Mac', 'iPad', 'App Store', 'Windows', 'Android', 'Microsoft Windows', 'PlayStation', 'Nintendo', 'Xbox', 'Epic Games, Inc.', 'Epic', 'Google LLC', 'U.S. Securities and Exchange Commission', 'SEC', 'The Nasdaq Stock Market LLC', 'S&P 500 Index', 'Dow Jones U.S. Technology Supersector Index', 'Department of Justice', 'DOJ', 'U.S. Internal Revenue Service', 'Board of Directors', 'Board', 'Audit Committee', 'General Court of the EU', 'European Commission', 'Commission', 'General Court', 'European Union', 'EU', 'Rule 10b5-1', 'Securities Exchange Act of 1934', 'Private Securities Litigation Reform Act of 1995', 'Digital Markets Act', 'DMA', 'Silicon Valley', 'COVID-19', 'U.S.

### ロジット後処理

In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
PROMPT = "Is Enzo a good name for a baby?"

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import LogitsProcessor, LogitsProcessorList
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
                                             torch_dtype=torch.bfloat16,
                                             device_map="auto")

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)

# ロジットを取得する
with torch.inference_mode():
    outputs = model(**inputs)
    logits = outputs.logits

# 最後のトークンに対応するロジット
last_token_logits = logits[:, -1, :]

next_token_probs = torch.nn.functional.softmax(last_token_logits, dim=-1)

k = 10
top_k_probs, top_k_indices = torch.topk(next_token_probs, k, dim=-1)

# 特殊トークンを除外して、実際のトークンを表示する
top_k_tokens = [
    tokenizer.decode(idx, skip_special_tokens=True)
    for idx in top_k_indices[0]
]

print("Top predicted tokens and probabilities:")
for prob, token in zip(top_k_probs[0][:k], top_k_tokens[:k]):
    if token.strip():  # 空でないトークンのみ表示する
        print(f"'{token}': {prob:.4f}")


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Top predicted tokens and probabilities:
' What': 0.0317
' I': 0.0317
' Here': 0.0193
' Is': 0.0103
' My': 0.0085


In [ ]:
class YesNoLogitsProcessor(LogitsProcessor):
    def __init__(self, yes, no, tokenizer, initial_length):
        self.yes = yes
        self.no = no
        self.tokenizer = tokenizer
        self.initial_length = initial_length

    def __call__(self, input_ids: torch.LongTensor,
                 scores: torch.FloatTensor) -> torch.FloatTensor:
        # すでに応答を生成している場合は、すべてのトークンをマスクする
        if len(input_ids[0]) > self.initial_length:
            scores.fill_(-float('inf'))
            return scores

        # デバッグ用の出力
        yes_tokens = self.tokenizer.encode(self.yes, add_special_tokens=False)
        no_tokens = self.tokenizer.encode(self.no, add_special_tokens=False)
        print(f"Yes token ID: {yes_tokens}")
        print(f"No token ID: {no_tokens}")

        # yes/noに対応する元のロジットを抽出する
        yes_no_logits = scores[:, [yes_tokens[0], no_tokens[0]]]
        print(f"[Yes, No] logits: {yes_no_logits}")

        # ソフトマックスを使用して確率を取得する
        yes_no_probs = torch.nn.functional.softmax(yes_no_logits, dim=-1)
        yes_prob = yes_no_probs[:, 0]
        no_prob = yes_no_probs[:, 1]
        print(f"Yes prob: {yes_prob}")
        print(f"No prob: {no_prob}")

        # すべてのトークンを-infでマスクする
        scores.fill_(-float('inf'))

        # 確率が高いほうの選択肢に大きな値を設定する
        yes_mask = yes_prob > no_prob
        high = scores.new_full((scores.shape[0],), 1e4)
        low = scores.new_full((scores.shape[0],), -float('inf'))
        scores[:, yes_tokens[0]] = torch.where(yes_mask, high, low)
        scores[:, no_tokens[0]] = torch.where(~yes_mask, high, low)

        return scores

In [ ]:
input_ids = tokenizer.encode(PROMPT, return_tensors="pt")
initial_length = len(input_ids[0])

YES = "yes"
NO = "no"

# 制御して生成する
generation_output_controlled = model.generate(
    **inputs,
    logits_processor=LogitsProcessorList([
        YesNoLogitsProcessor(YES, NO, tokenizer, initial_length)
    ]),
    max_length=50
)

# 制御せずに生成する
generation_output = model.generate(**inputs, max_length=50)

Yes token ID: [10407]
No token ID: [4607]
[Yes, No] logits: tensor([[2.6562, 3.0156]])
Yes prob: tensor([0.4111])
No prob: tensor([0.5889])


In [ ]:
generation_output

tensor([[ 4182,  1698, 15294,   253,  1123,  1462,   327,   253,  4516,    47,
           198,   198,  5056, 15294,   314,   253,  8170,  7423,  1462,   338,
           553,   719,   281,   722,   327,  5034,    30,   657,   314,  6435,
           429,   260,  5671,  1462,   476,  5056, 22992,  1627,   527,  1530,
           476,  1141,   325,  1837,    18,   355,   476,  1141,   325,  3269]])

In [ ]:
def generate_response(model_output, tokenizer):
    gen_output = tokenizer.batch_decode(model_output,
                                        skip_special_tokens=True,
                                        clean_up_tokenization_spaces=False)
    generated_text = gen_output[0][
                len(
                    tokenizer.decode(
                        inputs["input_ids"][0], skip_special_tokens=True
                    )
                ) :
            ].strip()
    return generated_text

In [ ]:
generate_response(generation_output_controlled, tokenizer)

'no'

In [ ]:
generate_response(generation_output, tokenizer)

'Enzo is a classic Italian name that has been in use for centuries. It is derived from the Latin name "Enetus," which means "to be strong" or "to be powerful'

### Outlines

In [ ]:
TOP = 1_000
prompt = f"""You are a sentiment-labeling assistant specialized
in Financial Statements.
Is the following document positive or negative?

Document: {sec_filing[:TOP]}
"""

In [ ]:
!pip install outlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.2/112.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.9 MB/s eta 0:00:00


In [ ]:
import torch
import outlines

from typing import Literal
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Hugging Faceのモデルとトークナイザを読み込む
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Outlines用のモデルに変換
model = outlines.from_transformers(
    hf_model,
    tokenizer
)
Sentiment = Literal["Positive", "Negative"]

answer = model(
    prompt,
    output_type=Sentiment,
    max_new_tokens=10
)

answer

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

'Positive'

In [ ]:
BASE_PROMPT = """
You are an expert at structured data extraction.
You will be given unstructured text from an SEC filing.
Extract the mentioned entities and places and return the result
using the specified structure.
""".strip()

prompt = f"""
{BASE_PROMPT}

Document:
{sec_filing[:TOP]}
""".strip()

sec_extraction_json = model(
    prompt,
    output_type=SECExtraction,
    max_new_tokens=500
)

sec_extraction_outlines = SECExtraction.model_validate_json(
    sec_extraction_json
)
print("Extracted entities:", sec_extraction_outlines.mentioned_entities)
print("Extracted places:", sec_extraction_outlines.mentioned_places)
sec_extraction_outlines

Extracted entities: ['Apple Inc.', 'SEC', 'Fiscal Year Ended September 28, 2024']
Extracted places: ['Form 10-K', 'Business', 'Risk Factors', 'Unresolved Staff Comments', 'Cybersecurity', 'Properties', 'Legal Proceedings', 'Mine Safety Disclosures', 'Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'Financial Statements and Supplementary Data', 'Quantitative and Qualitative Disclosures About Market Risk', 'Controls and Procedures', 'Other Information', 'Foreign Jurisdictions that Prevent Inspections', 'Directors, Executive Officers and Corporate Governance', 'Executive Officers']


SECExtraction(mentioned_entities=['Apple Inc.', 'SEC', 'Fiscal Year Ended September 28, 2024'], mentioned_places=['Form 10-K', 'Business', 'Risk Factors', 'Unresolved Staff Comments', 'Cybersecurity', 'Properties', 'Legal Proceedings', 'Mine Safety Disclosures', 'Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities', 'Financial Statements and Supplementary Data', 'Quantitative and Qualitative Disclosures About Market Risk', 'Controls and Procedures', 'Other Information', 'Foreign Jurisdictions that Prevent Inspections', 'Directors, Executive Officers and Corporate Governance', 'Executive Officers'])